In [1]:
import numpy as np
import matplotlib.pyplot as plt
import os
from astropy.io import fits
from astropy.table import Table
from astropy.table import vstack

%matplotlib inline
import os
import warnings
warnings.filterwarnings('ignore')

import scipy.ndimage as nd

import astropy.io.fits as pyfits
import astropy.units as u

import sep

import grizli
from grizli import utils
print(f'grizli version: {grizli.__version__}')

import msaexp
import msaexp.spectrum
print(f'msaexp version: {msaexp.__version__}')

BASE_URL = 'https://s3.amazonaws.com/msaexp-nirspec/extractions/'



In [2]:
def open_fits(path, ext):
    
    with fits.open(path) as hdul:

        tab = Table(hdul[ext].data)

    return tab

In [3]:
TABLE = open_fits("/nvme/scratch/work/alberttg/Summer_project/Ha_and_NII_broad_line_data.fits", ext=1)

FILES = TABLE["DJA_file"] # String names of the fits files in the DJA database
GALAXY_ID = TABLE["SURVEY_ID"]
REDSHIFT = TABLE["REDSHIFT"]
ROOTS = TABLE["src_id"] # Source ID which I think is root



FileNotFoundError: [Errno 2] No such file or directory: 'rubies-uds42-v3_g395m-f290lp_4233_23438.spec.fits'

In [ ]:
def full_summary_catalogue()

    nrs = utils.read_catalog(BASE_URL + 'nirspec_graded_v0.ecsv', format='ascii.ecsv')
    
    print('# Grade')
    un = utils.Unique(nrs['grade'])
    
    # By grating
    grat = utils.Unique(nrs['grating'])
    
    # By project
    root = utils.Unique(nrs['root'])
    
    # Robust redshifts
    robust_prism = (nrs['grating'] == 'PRISM') & (nrs['grade'] == 3)

    roots = utils.Unique(nrs['root'][robust_prism], verbose=False)

    nx = 2
    ny = int(np.ceil((len(roots) + 1) / 2))

    sx = 2.5

    fig, axes = plt.subplots(ny,nx,sharex=True, sharey=False, figsize=(nx*sx*2, ny*sx))

    lnz = np.log(1+nrs['z'])

    bins = utils.log_zgrid([0.1, 13], 0.1)

    xtv = [0,1,2,3,4,5,6,7,8,9,10,11,12,13]
    xtl = [0,1,2,3,4,5,6,7,8,9,10,'',12,'']

    _ = axes[0][0].hist(lnz[robust_prism], bins=np.log(1+bins))
    axes[0][0].set_ylabel('All PRISM')
    axes[0][0].grid()

    count = 0

    for root in roots.values:
        count += 1
        i = count // ny
        j = count - ny*i

        _ = axes[j][i].hist(lnz[robust_prism][roots[root]], bins=np.log(1+bins))
        axes[j][i].set_ylabel(root)
        axes[j][i].grid()

    axes[j][i].set_xticks(np.log(1+np.array(xtv)))
    axes[j][i].set_xticklabels(xtl)

    for last in range(count*1, nx*ny-1):
        count += 1
        i = count // ny
        j = count - ny*i

        _ = axes[j][i].axis('off')

    _ = fig.tight_layout(pad=1)

In [ ]:

def detect_oiii_doublet(ax, sp, z, threshold=5.0, window=0.04):
    """
    Detect the [OIII]4959,5007 doublet by fitting two Gaussians.

    Parameters
    ----------
    ax: axes
        Plotting axes
    sp : SpectrumSampler
        JWST spectrum.
    z : float
        Galaxy redshift.
    threshold : float
        Detection significance.
    window : float
        Half-width (microns) around 5007.

    Returns
    -------
    detected : bool
    snr : float
        S/N of the fitted 5007 line.
    popt : ndarray
        Best-fit parameters.
    pcov : ndarray
        Covariance matrix.
    """

    # Rest wavelengths (microns)
    lam5007 = 0.500684 * (1 + z)
    lam4959 = 0.496030 * (1 + z)

    wave = sp.spec_wobs
    flux = sp.spec["flux"]
    err = sp.spec["full_err"]

    mask = (
        (wave > lam4959 - window) &
        (wave < lam5007 + window) &
        sp.valid
    )

    wave = wave[mask]
    flux = flux[mask]
    err = err[mask]

    if len(wave) < 10:
        return False, 0.0, None, None

    def model(x, continuum, amp5007, sigma):
        """
        amp5007 = peak amplitude of λ5007.
        λ4959 amplitude fixed to amp5007/2.98.
        """

        gauss = amp5007 * np.exp(
            -(x - lam5007)**2 / (2*sigma**2)
        )

        g4959 = (amp5007/2.98) * np.exp(
            -(x - lam4959)**2 / (2*sigma**2)
        )

        return continuum + g4959 + g5007

    # Initial guesses
    continuum0 = np.median(flux)
    amp0 = np.max(flux) - continuum0
    sigma0 = 0.003       # μm

    p0 = [continuum0, amp0, sigma0]

    try:

        popt, pcov = curve_fit(
            model,
            wave,
            flux,
            p0=p0,
            sigma=err,
            absolute_sigma=True,
            maxfev=10000
        )

    except RuntimeError:
        return False, 0.0, None, None

    continuum, amp5007, sigma = popt

    # Integrated flux of λ5007
    flux5007 = amp5007 * sigma * np.sqrt(2*np.pi)

    # Error propagation
    amp_err = np.sqrt(pcov[1,1])
    sigma_err = np.sqrt(pcov[2,2])

    flux_err = flux5007 * np.sqrt(
        (amp_err/amp5007)**2 +
        (sigma_err/sigma)**2
    )

    snr = flux5007 / flux_err

    detected = snr >= threshold
    
    ax.plot(wave, model(continuum, amp5007, sigma), alpha=0.8, label('[OIII] fit')

    return detected, snr, popt, pcov

In [ ]:
def read_galaxy_source_from_catalogue(root, file):
    """
    This function is for anythin missing that Shay didn't give me.
    """
    path_to_file = BASE_URL + "{root}/{file}"
    
    nrs = utils.read_catalog(BASE_URL + 'nirspec_graded_v0.ecsv', format='ascii.ecsv')
    
    is_gds = np.array([f.startswith('gds-deep') for f in nrs['file']]) #gds-deep is an example

    src = np.array([id in f for f in nrs['file']]) & is_gds

    nrs['root','file','grating','filter','z','grade'][src]

    urls = [path_to_file.format(**row) for row in nrs[src]]

    z = np.mean(nrs['z'][src & (nrs['grade'] == 3)])

    nrs[src]['root','file','grating','filter','z','grade']
    # Table with relevant info for a specific galaxy
    
    return urls

In [ ]:
def make_urls(root, files):
    """
    Construct DJA URLs for one or more extracted spectra.

    Parameters
    ----------
    root : str
    files : str or list of str
        One or more *.spec.fits filenames.

    Returns
    -------
    urls : list of str
    """

    if isinstance(files, str):
        files = [files]

    path_to_file = BASE_URL + "{root}/{file}"

    return [
        path_to_file.format(root=root, file=f)
        for f in files
    ]

In [ ]:
def plot_spectra(urls):
    
    for u in urls:
    if 'prism' in u:
        break

    print(f'Load 2D file: {u}')

    img = pyfits.open(u)
    spec = utils.GTable(img['SPEC1D'].data)
    img.info()
    
    """
    # Metadata and exposure data
    for i, k in enumerate(img['SCI'].header):
        print(f"{k} : {img['SCI'].header[k]}")
        if k == 'FILE2':
            break

    print('...')
    """
    #-----------------------------------------------------------------------------------------------------
    # 2D raw spectra
    fig, axes = plt.subplots(3,1, figsize=(8, 5), sharex=True, sharey=True)

    msk = img['WHT'].data > 0

    for i, k in enumerate(['SCI','WHT','PROFILE']):
        _data = img[k].data
        vm = np.percentile(_data[msk], [2,99])
        ax = axes[i]
        ax.imshow(_data, vmin=vm[0], vmax=vm[1], cmap='plasma_r', aspect='auto')
        ax.set_ylabel(k)
        ax.set_yticklabels([])

    xtv = [1,2,3,4,5]
    xti = np.interp(xtv, spec['wave'], np.arange(len(spec)))

    a2 = axes[0].twiny()
    a2.set_xlabel('pix')
    a2.set_xlim(0, _data.shape[1])

    ax.set_xticks(xti)
    ax.set_xticklabels(xtv)

    ax.set_xlabel(r'$\lambda_\mathrm{obs}\,[\mu\mathrm{m}]$')

    _ = fig.tight_layout(pad=1)

        
    #---------------------------------------------------------------------------------------------------------
    # Optimal extraction profile
    # Collapsed 1D profile
    p1 = utils.GTable(img['PROF1D'].data)

    fig, ax = plt.subplots(1,1,figsize=(5,3))
    ax.plot(p1['pix'], p1['profile'], label='data', color='k')
    ax.plot(p1['pix'], p1['pfit'], label='fit', color='r', lw=4, alpha=0.5)

    ax.set_xlabel(r'$\Delta y$, pix')
    ax.set_ylabel('Collapsed profile')
    ax.legend()

    ax.grid()
    #---------------------------
    wlims = [(1.8, 2.1), (5, 5.2)]
    fig, ax = plt.subplots(1, 1, figsize=(8, 4), sharex=True)

    w2d = np.ones(img['SCI'].data.shape)*spec['wave']

    for i, wlim in enumerate(wlims):
        msk = (w2d > wlim[0]) & (w2d < wlim[1]) & (img['WHT'].data > 0)
        ydata = (img['SCI'].data*msk).sum(axis=1)
        pdata = (img['PROFILE'].data*msk).sum(axis=1)

        anorm = (ydata*pdata).sum()/(pdata**2).sum()
        pdata *= anorm
        norm = pdata.sum()

        pl = ax.plot(ydata/norm, label=r'$xx < \lambda < yy$'.replace('xx', f'{wlim[0]:.1f}').replace('yy', f'{wlim[1]:.1f}'))
        ax.plot(pdata/norm, label='profile', color=pl[0].get_color(), linestyle='--')

    ax.set_ylim(-0.18, 0.5)
    ax.legend()
    ax.set_xlim(10,30)
    ax.grid()


In [ ]:
def OIII_model(x, continuum, amp5007, sigma, lam5007, lam4959):
        """
        amp5007 = peak amplitude of λ5007.
        λ4959 amplitude fixed to amp5007/2.98.
        """

        gauss = amp5007 * np.exp(
            -(x - lam5007)**2 / (2*sigma**2)
        )

        g4959 = (amp5007/2.98) * np.exp(
            -(x - lam4959)**2 / (2*sigma**2)
        )

        return continuum + g4959 + g5007

In [ ]:
def oneD_spectra(urls, z):
    
    # [OIII] doublet obs wavelengths (microns)
    lam5007 = 0.500684 * (1 + z)
    lam4959 = 0.496030 * (1 + z)
    
    # Ha observed emission line (microns)
    Ha_obs = 0.65646 * (1+z)
    
    # Open the files into 1D objects, directly from the web
    sobj = {}
    for u in urls:
        print(f'Read {u}')
        key = os.path.basename(u)
        sobj[key] = msaexp.spectrum.SpectrumSampler(u)
        
    # A single (prism) spectrum

    for i, k in enumerate(sobj):
        if 'prism' in k:
            break

    sp = sobj[k]
    sp.spec.info()
    
    """
    # Metadata about the 1D extraction
    for i, k in enumerate(sp.spec.meta):
        print(f"{k} : {sp.spec.meta[k]}")
        if k == 'CRDS2':
            break

    print('...')
    """    
    
    fig, axes = plt.subplots(2,1,figsize=(8,6), sharex=True)
    
    ax = axes[0]

    for c in ['flux','err','full_err']:
        ax.plot(sp.spec_wobs, sp.spec[c], alpha=0.5, label=c)
    #ax.plot(sp.spec_wobs, sp.spec['err'], alpha=0.5, label='err')
    #ax.plot(sp.spec_wobs, sp.spec['full_err'], alpha=0.5, label='full_err')
    
    detected, snr, popt, pcov = detect_oiii(sp, z)#--------------------------OIII fit
    
    continuum, amp5007, sigma = popt
    
    print(f"[OIII] S/N = {snr:.2f}")

    ax.set_ylim(-0.1*ymax[1], ymax[1])
    ax.plot(sp.spec_wobs, sp.valid*ymax[1]*0.8, alpha=0.5, label='"valid"')
    ax.grid()
    ax.legend()
    ax.set_ylabel(r'$F_\nu\,[\mu\mathrm{Jy}]$')

    ax = axes[1]
    for c in ['flux','err','full_err']:
        ax.plot(sp.spec_wobs, sp.spec[c]*sp.spec['to_flam'], alpha=0.5, label=c)

    ax.set_ylim(-0.1*ymax[1]*5, ymax[1]*5)
    ax.set_ylabel(r'$F_\lambda\,[10^{-20}\mathrm{erg/s/cm2/A}]$')
    ax.grid()

    ax.set_xlabel(r'$\lambda_\mathrm{obs}\,[\mu\mathrm{m}]$')

    _ = fig.tight_layout(pad=1)
    
    
    #---------------------------------------------------------------------------------------------------------
    # Show all spectra
    fig, axes = plt.subplots(len(xlimits),1,figsize=(8,3*len(xlimits)))

    for ax, xlim, ym in zip(axes, xlimits, ymax):
        for i, k in enumerate(sobj):
            sp = sobj[k]
            ax.plot(sp.spec_wobs[sp.valid], sp.spec['flux'][sp.valid], alpha=0.5, label=k.split('_')[1])

        ax.set_xlim(*xlim)

        ax.set_ylim(-0.1*ym, ym)
        ax.grid()

    axes[0].legend()
    ax.set_xlabel(r'$\lambda_\mathrm{obs}\,[\mu\mathrm{m}]$')

    _ = fig.tight_layout(pad=1)
    
    return detected, snr

In [ ]:
if __name__ == "__main__":
    
    for i in range(len(FILES)):
        
        file = FILES[i]
        root = ROOTS[i]
        redshift = REDSHIFT[i]
        Id = GALAXY_ID[i]
    
        urls = make_urls(root, file)
        # plot_spectra(urls)
        oneD_spectra(urls, redshift)
    